# Tải dataset từ Roboflow và cắt object theo nhãn + theo tập (train / val / test)

Project: `nguyen-minh-hieu-ev5ll/violence-bblcr`

Kết quả cắt được lưu theo cấu trúc:
```
cropped/
├── train/
│   ├── violence/       *.jpg
│   └── non-violence/   *.jpg
├── val/
└── test/
```

## 1. Cài thư viện

In [ ]:
!pip -q install roboflow opencv-python-headless pyyaml tqdm pandas

## 2. Cấu hình

In [ ]:
import os, getpass

# ===== CẤU HÌNH =====
WORKSPACE = "nguyen-minh-hieu-ev5ll"
PROJECT   = "violence-bblcr"
VERSION   = None        # None = tự lấy version mới nhất; hoặc điền số, ví dụ 1
FORMAT    = "yolov8"    # "yolov8" (mặc định) hoặc "coco"

DATASET_DIR = "/content/dataset"   # nơi tải dataset gốc về
CROP_DIR    = "/content/cropped"   # nơi lưu ảnh đã cắt

PAD_RATIO = 0.0   # nới bbox thêm x% mỗi phía (vd 0.1 = 10%), 0 = cắt đúng khít bbox
MIN_CROP  = 224   # cạnh tối thiểu của ảnh crop (224 = input chuẩn của EfficientNet-B0 / MobileNet / ResNet-18)
PAD_MODE  = "border"  # cách xử lý box nhỏ hơn MIN_CROP:
                      #   "border"  = cắt ĐÚNG khít bbox, rồi đệm viền xám (căn giữa) cho đủ MIN_CROP
                      #   "context" = nới bbox lấy thêm vùng ảnh xung quanh cho đủ MIN_CROP
SQUARE    = False # True = ép mọi crop thành hình vuông (đỡ méo ảnh khi resize)
MIN_BOX   = 2     # bỏ box lỗi có rộng/cao nhỏ hơn số pixel này

SAVE_TO_DRIVE   = True
DRIVE_DIR       = "/content/drive/MyDrive/violence_crops"   # = /content/drive/MyDrive/ của Google Drive cá nhân
OVERWRITE_DRIVE = True   # True = xóa thư mục DRIVE_DIR cũ (nếu có) trước khi copy để không lẫn file cũ

# API key: ưu tiên lấy từ Colab Secrets (biểu tượng chìa khóa bên trái, tên ROBOFLOW_API_KEY)
try:
    from google.colab import userdata
    API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    API_KEY = getpass.getpass("Nhập Roboflow API key: ")

## 3. Tải dataset bằng Roboflow API

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)

if VERSION is None:
    try:
        VERSION = max(int(v.id.split("/")[-1]) for v in project.versions())
    except Exception:
        VERSION = 1
print("Dùng version:", VERSION)

dataset = project.version(VERSION).download(FORMAT, location=DATASET_DIR)
print("Đã tải về:", dataset.location)

## 4. Hàm đọc annotation và cắt object

In [ ]:
import json, re, cv2, yaml, shutil
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.auto import tqdm

SPLIT_MAP = {"train": "train", "valid": "val", "val": "val", "test": "test"}  # Roboflow gọi val là "valid"
IMG_EXT = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]


def safe(name):
    return re.sub(r"[^\w\-]+", "_", str(name)).strip("_")


def n_files(d):
    return sum(1 for f in Path(d).rglob("*") if f.is_file())


def expand_box(x1, y1, x2, y2, W, H):
    """PAD_MODE="border": trả về đúng bbox (đã kẹp trong ảnh).
    PAD_MODE="context": nới box (lấy vùng ảnh xung quanh) cho đủ MIN_CROP, chạm mép ảnh thì dịch cửa sổ vào trong."""
    pw, ph = (x2 - x1) * PAD_RATIO, (y2 - y1) * PAD_RATIO
    x1, y1, x2, y2 = x1 - pw, y1 - ph, x2 + pw, y2 + ph
    if PAD_MODE == "border":                        # cắt khít bbox, việc đệm cho đủ MIN_CROP làm ở crop_and_save
        return max(0, int(round(x1))), max(0, int(round(y1))), min(W, int(round(x2))), min(H, int(round(y2)))
    tw, th = max(x2 - x1, MIN_CROP), max(y2 - y1, MIN_CROP)
    if SQUARE:
        tw = th = max(tw, th)
    tw, th = min(tw, W), min(th, H)                 # không vượt quá kích thước ảnh gốc
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    nx1 = min(max(cx - tw / 2, 0), W - tw)
    ny1 = min(max(cy - th / 2, 0), H - th)
    return int(round(nx1)), int(round(ny1)), int(round(nx1 + tw)), int(round(ny1 + th))


def crop_and_save(img, box_xyxy, out_path):
    """Trả về None (box lỗi, bỏ qua), "ok" hoặc "expanded" (box nhỏ, đã nới thêm)."""
    H, W = img.shape[:2]
    x1, y1, x2, y2 = box_xyxy
    if (x2 - x1) < MIN_BOX or (y2 - y1) < MIN_BOX:
        return None
    small = (x2 - x1) < MIN_CROP or (y2 - y1) < MIN_CROP

    ex1, ey1, ex2, ey2 = expand_box(x1, y1, x2, y2, W, H)
    crop = img[ey1:ey2, ex1:ex2]
    if crop.size == 0:                               # box nằm hẳn ngoài ảnh
        return None

    # Đệm viền xám (căn giữa) nếu crop còn nhỏ hơn MIN_CROP (hoặc cần vuông)
    ch, cw = crop.shape[:2]
    tw, th = max(cw, MIN_CROP), max(ch, MIN_CROP)
    if SQUARE:
        tw = th = max(tw, th)
    if (tw, th) != (cw, ch):
        dx, dy = tw - cw, th - ch
        crop = cv2.copyMakeBorder(crop, dy // 2, dy - dy // 2, dx // 2, dx - dx // 2,
                                  cv2.BORDER_CONSTANT, value=(114, 114, 114))

    out_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_path), crop)
    return "expanded" if small else "ok"


def iter_coco(split_dir):
    """Yield (đường dẫn ảnh, [(nhãn, bbox_xyxy_pixel)], normalized=False)"""
    data = json.load(open(split_dir / "_annotations.coco.json", encoding="utf-8"))
    cats = {c["id"]: c["name"] for c in data["categories"]}
    imgs = {i["id"]: i for i in data["images"]}
    per_img = defaultdict(list)
    for a in data["annotations"]:
        x, y, w, h = a["bbox"]
        per_img[a["image_id"]].append((cats[a["category_id"]], (x, y, x + w, y + h)))
    for iid, anns in per_img.items():
        yield split_dir / imgs[iid]["file_name"], anns, False


def iter_yolo(split_dir, names):
    """Yield (đường dẫn ảnh, [(nhãn, bbox_xyxy_chuẩn_hóa)], normalized=True)"""
    img_dir, lbl_dir = split_dir / "images", split_dir / "labels"
    for lp in lbl_dir.glob("*.txt"):
        ip = next((img_dir / (lp.stem + e) for e in IMG_EXT if (img_dir / (lp.stem + e)).exists()), None)
        if ip is None:
            continue
        anns = []
        for line in lp.read_text().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            cls, vals = int(parts[0]), list(map(float, parts[1:]))
            if len(vals) == 4:                      # bbox: xc yc w h
                xc, yc, w, h = vals
                box = (xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2)
            else:                                   # polygon -> bbox
                xs, ys = vals[0::2], vals[1::2]
                box = (min(xs), min(ys), max(xs), max(ys))
            anns.append((names[cls], box))
        if anns:
            yield ip, anns, True

## 5. Cắt object theo nhãn và theo từng tập

In [ ]:
root = Path(DATASET_DIR)
crop_root = Path(CROP_DIR)
if crop_root.exists():
    shutil.rmtree(crop_root)

names = None
if FORMAT != "coco":
    n = yaml.safe_load(open(root / "data.yaml", encoding="utf-8"))["names"]
    names = [n[k] for k in sorted(n)] if isinstance(n, dict) else n

stats, expanded, skipped = Counter(), Counter(), 0
split_dirs = sorted(d for d in root.iterdir() if d.is_dir() and d.name in SPLIT_MAP)
print("Các tập tìm thấy:", [d.name for d in split_dirs])

for sd in split_dirs:
    split = SPLIT_MAP[sd.name]
    items = list(iter_coco(sd) if FORMAT == "coco" else iter_yolo(sd, names))
    for ip, anns, normalized in tqdm(items, desc=split):
        img = cv2.imread(str(ip))
        if img is None:
            continue
        H, W = img.shape[:2]
        for k, (label, box) in enumerate(anns):
            if normalized:
                box = (box[0] * W, box[1] * H, box[2] * W, box[3] * H)
            out = crop_root / split / safe(label) / f"{ip.stem}_{k}.jpg"
            res = crop_and_save(img, box, out)
            if res is None:
                skipped += 1
                continue
            stats[(split, label)] += 1
            if res == "expanded":
                expanded[(split, label)] += 1

import pandas as pd
df = pd.DataFrame([(s, l, c) for (s, l), c in stats.items()], columns=["split", "label", "so_luong"])
print(df.pivot_table(index="label", columns="split", values="so_luong", aggfunc="sum", fill_value=0, margins=True))

tot, exp = sum(stats.values()), sum(expanded.values())
print(f"\nTổng {tot} crop, trong đó {exp} crop ({exp / max(tot, 1):.1%}) là box nhỏ đã được nới padding lên >= {MIN_CROP}px")
print(f"Bỏ qua {skipped} box lỗi (rộng/cao < {MIN_BOX}px)")

## 6. Thống kê chi tiết sau xử lý

In [ ]:
from PIL import Image
from IPython.display import display

crop_root = Path(CROP_DIR)
print(f"{crop_root}  —  TỔNG: {n_files(crop_root)} file\n")

# --- Số file trong từng thư mục ---
for sp in sorted(d for d in crop_root.iterdir() if d.is_dir()):
    print(f"{sp.name}/  —  {n_files(sp)} file")
    for lb in sorted(d for d in sp.iterdir() if d.is_dir()):
        print(f"    {lb.name}/  —  {n_files(lb)} file")

# --- Đọc kích thước từng ảnh (chỉ đọc header nên nhanh) ---
recs = []
for f in crop_root.rglob("*"):
    if f.is_file() and f.suffix.lower() in IMG_EXT:
        split, label = f.relative_to(crop_root).parts[:2]
        with Image.open(f) as im:
            w, h = im.size
        recs.append((split, label, w, h))
df_all = pd.DataFrame(recs, columns=["split", "label", "w", "h"])

order = [c for c in ["train", "val", "test"] if c in set(df_all.split)]

# --- Bảng số lượng theo nhãn x tập ---
cnt = df_all.pivot_table(index="label", columns="split", values="w", aggfunc="count", fill_value=0)[order]
cnt["Tổng"] = cnt.sum(axis=1)
cnt.loc["Tổng"] = cnt.sum()
print("\n== Số lượng file (nhãn x tập) ==")
display(cnt)

# --- Tỷ lệ % mỗi nhãn trong từng tập ---
print("== Tỷ lệ nhãn trong mỗi tập (%) ==")
display((cnt.iloc[:-1] / cnt.iloc[-1] * 100).round(1))

# --- Tỷ lệ phân chia train / val / test ---
print("== Tỷ lệ phân chia các tập (%) ==")
display((cnt.loc["Tổng", order] / cnt.loc["Tổng", "Tổng"] * 100).round(1).to_frame("%").T)

# --- Kích thước ảnh ---
print("== Kích thước ảnh (rộng, cao) theo tập ==")
display(df_all.groupby("split")[["w", "h"]].agg(["min", "mean", "max"]).round(0).astype(int).loc[order])
print("Số ảnh có cạnh < MIN_CROP:", int(((df_all.w < MIN_CROP) | (df_all.h < MIN_CROP)).sum()))

## 7. Xem thử một vài ảnh đã cắt

In [ ]:
import random
import matplotlib.pyplot as plt

def preview(split="train", n=5):
    base = Path(CROP_DIR) / split
    if not base.exists():
        print("Không có tập", split); return
    labels = sorted(p.name for p in base.iterdir() if p.is_dir())
    fig, axes = plt.subplots(len(labels), n, figsize=(3 * n, 3 * len(labels)), squeeze=False)
    for r, lb in enumerate(labels):
        fs = list((base / lb).glob("*.jpg"))
        random.shuffle(fs)
        for c in range(n):
            ax = axes[r][c]; ax.axis("off")
            if c < len(fs):
                ax.imshow(cv2.cvtColor(cv2.imread(str(fs[c])), cv2.COLOR_BGR2RGB))
                if c == 0:
                    ax.set_title(lb, loc="left")
    plt.tight_layout(); plt.show()

preview("train")

## 8. Lưu kết quả vào Google Drive cá nhân

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    dst = Path(DRIVE_DIR)
    if OVERWRITE_DRIVE and dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(CROP_DIR, dst, dirs_exist_ok=True)

    # thêm 1 bản zip cạnh thư mục (tải/upload lại nhanh hơn nhiều file nhỏ)
    zip_path = shutil.make_archive(str(dst.parent / dst.name), "zip", CROP_DIR)

    print("Đã lưu vào Drive:", dst)
    print("File zip        :", zip_path)
    print(f"Kiểm tra: {n_files(CROP_DIR)} file (Colab) -> {n_files(dst)} file (Drive)")
else:
    print("SAVE_TO_DRIVE = False, bỏ qua bước lưu vào Drive.")